# Load deps

In [ ]:
# ! pip install -q torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
import sys
app_path = '/content/drive/MyDrive/Projects/miniSD'
sys.path.append(app_path)

In [ ]:
import os,math,torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from types import SimpleNamespace
from functools import partial
from datasets import load_dataset
import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from diffusers import UNet2DModel

from src.utils import set_seed
from src.datasets import inplace, DataLoaders, show_images
from src.learner import DeviceCB, ProgressCB, MetricsCB, Learner
from src.accel import MixedPrecision
from src.sgd import BatchSchedCB

# Config

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES']='1'
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
set_seed(42)
num_dl_workers = os.cpu_count()
dataset_xl,dataset_yl = 'image','label'
dataset_name = "fashion_mnist"
bs = 128
lr = 1e-2
epochs = 10
n_steps = 400
# we decrease bs, epochs, and n_steps for faster training and sampling

# Load dataset

The most impotant chaage in this DDPM-V3 is that we use [-0.5,0.5] input range instead of [-1,1] or [0,1]
- it's symmetrical
- it also yields better result than [-1,1]
    - [-1,1] is what everybody does, but there's no theoretical reason behind it!!! 

In [ ]:
@inplace
def transformi(b):
    b[dataset_xl] = [
        F.pad(TF.to_tensor(o), (2,2,2,2))-0.5
        for o in b[dataset_xl]
    ]

dsd = load_dataset(dataset_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)

# Train components

## Cosine scheduler

Another important change is to use cosine scheduler instead of the linear one

In [ ]:
def linear_sched(betamin=0.0001,betamax=0.02,n_steps=n_steps):
    beta = torch.linspace(betamin, betamax, n_steps)
    return SimpleNamespace(a=1.-beta, abar=(1.-beta).cumprod(dim=0), sig=beta.sqrt())

In [ ]:
def abar(t, T): return (t/T*math.pi/2).cos()**2
def cos_sched(n_steps=n_steps):
    ts = torch.linspace(0, n_steps-1, n_steps)
    ab = abar(ts,n_steps)
    alp = ab/abar(ts-1,n_steps)
    return SimpleNamespace(a=alp, abar=ab, sig=(1-alp).sqrt())

In [ ]:
lin_abar = linear_sched(betamax=0.02).abar
# by descreasing betamax (or n_steps from 1000 to 400) we would have a curve more similar to the cosine curve
cos_abar = cos_sched().abar
fig, axs = plt.subplots(1,2, figsize=(12,4))
axs[0].plot(lin_abar, label='lin')
axs[0].plot(cos_abar, label='cos')
axs[0].set_title("alpha bar")
axs[0].legend();
axs[1].plot(lin_abar[1:]-lin_abar[:-1], label='lin')
axs[1].plot(cos_abar[1:]-cos_abar[:-1], label='cos')
axs[1].set_title("alpha bar diff")
axs[1].legend();

In [ ]:
# we're keeping the linear scheduler for V3 but decrease betamax to 0.01
# or keep betamax = 0.02 and n_steps = 400 for faster training
# lin_abar = linear_sched(betamax=0.01)
lin_abar = linear_sched()
alphabar = lin_abar.abar
alpha = lin_abar.a
sigma = lin_abar.sig

## Noisifier

In [ ]:
def noisify(x0, ᾱ):
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    ε = torch.randn(x0.shape, device=device)
    ᾱ_t = ᾱ[t].reshape(-1, 1, 1, 1).to(device)
    xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε
    return (xt, t.to(device)), ε

In [ ]:
dt = dls.train
xb,yb = next(iter(dt))
(xt,t),ε = noisify(xb[:25],alphabar)
show_images(xt[:25], imsize=1.5, titles=list(t.cpu().numpy()));

# Training

In [ ]:
class UNet(UNet2DModel):
    def forward(self, x): return super().forward(*x).sample

def init_ddpm(model):
    for o in model.down_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()
            if o.downsamplers:
                for p in list(o.downsamplers): init.orthogonal_(p.conv.weight)
    for o in model.up_blocks:
        for p in o.resnets: p.conv2.weight.data.zero_()
    model.conv_out.weight.data.zero_()

def collate_ddpm(b):
    return noisify(default_collate(b)[dataset_xl], alphabar)
def dl_ddpm(ds, nw=num_dl_workers):
    return DataLoader(ds, batch_size=bs, collate_fn=collate_ddpm, num_workers=nw)

Two other major difference compared to DDPM_v2:
- Bigger model (doubled the # channels)
- Trained for longer (10 epochs instead of 6)

In [ ]:
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))
opt_func = partial(optim.AdamW, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [
    DeviceCB(), MixedPrecision(),
    ProgressCB(plot=True), MetricsCB(),
    BatchSchedCB(sched)
]
model = UNet(
    in_channels=1, out_channels=1,
    block_out_channels=(32, 64, 128, 256), norm_num_groups=8
)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
# learn.fit(epochs)

In [ ]:
mdl_path = Path('artifacts/models')
mdl_path.mkdir(exist_ok=True, parents=True)
# upload artifacts directory to colab if needed

In [ ]:
# torch.save(learn.model, mdl_path/'fashion_ddpm3_mp.pkl')
model_art = torch.load(mdl_path/'fashion_ddpm3_mp.pkl', weights_only=False).cuda()
model.load_state_dict(model_art.state_dict())
learn.model = model

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def sample(model, sz):
    ps = next(model.parameters())
    x_t = torch.randn(sz).to(ps)
    preds = []
    for t in tqdm(reversed(range(n_steps)), total=n_steps):
        t_batch = torch.full((x_t.shape[0],), t, device=ps.device, dtype=torch.long)
        z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(ps)
        ᾱ_t1 = alphabar[t-1]  if t > 0 else torch.tensor(1)
        b̄_t = 1-alphabar[t]
        b̄_t1 = 1-ᾱ_t1
        noise = model((x_t, t_batch))
        x_0_hat = ((x_t - b̄_t.sqrt() * noise)/alphabar[t].sqrt())
        x_t = x_0_hat * ᾱ_t1.sqrt()*(1-alpha[t])/b̄_t + x_t * alpha[t].sqrt()*b̄_t1/b̄_t + sigma[t]*z
        preds.append(x_t.float().cpu())
    return preds

In [ ]:
n_samples = bs
# FID is biased against the sample size
# So, we use this to match the batch size so we can compare this to real images
samples = sample(model, (n_samples, 1, 32, 32))

In [ ]:
s = (samples[-1]*2)#.clamp(-1,1)
print(s.min(),s.max())
show_images(s[:16], imsize=1.5);

# Evaluate V3 model

## Load the reference model (the hard way!!!)

In [ ]:
from torch import distributions, tensor
from src.init import GeneralRelu, init_weights
from src.resnet import ResBlock
from src.augment import capture_preds
# TODO: we use this import to patch the capture_preds to the `Learner` class
# this is not a good practice at all. change it ASAP

class Dropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training: return x
        dist = distributions.binomial.Binomial(tensor(1.0).to(x.device), probs=1-self.p)
        return x * dist.sample(x.size()) * 1/(1-self.p)

def get_dropmodel(
    act=nn.ReLU,
    nfs=(16,32,64,128,256,512),
    norm=nn.BatchNorm2d,
    drop=0.0
):
    layers = [
        ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm),
        nn.Dropout2d(drop)
    ]
    layers += [
        ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
        for i in range(len(nfs)-1)
    ]
    layers += [
        nn.Flatten(),
        Dropout(drop),
        nn.Linear(nfs[-1], 10, bias=False),
        nn.BatchNorm1d(10)
    ]
    return nn.Sequential(*layers)

act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
cmodel = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
loaded_art = torch.load(mdl_path / 'data_aug2.pkl', weights_only=False)
cmodel.load_state_dict(loaded_art.state_dict())
cmodel = cmodel[:-3] # the desired feature head

In [ ]:
from src.fid import ImageEval

@inplace
def transformi2(b): b[dataset_xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))*2-1 for o in b[dataset_xl]]
# the transformation should be exactly the same as the one used to train the reference model

tds2 = dsd.with_transform(transformi2)
dls2 = DataLoaders.from_dd(tds2, bs, num_workers=num_dl_workers)
ie = ImageEval(cmodel, dls2, cbs=[DeviceCB()])

## Measure and compare FID

In [ ]:
print(s.min(),s.max(), s.shape)
print(xb.min(),xb.max(), xb.shape)

In [ ]:
print(ie.fid(s))
print(ie.fid(xb*2))

- this might mean that in terms of quality for an unconditional image generation pipeline, we're done
- considering tha fact that in comparison to the original fast-ai notebook, in order to sample and train faster we used:
    - n_steps = 400 instead of 1000
    - epochs = 10 instead of 25
    - bs = 128 instead of 512, which not have so much effect unlike other two configs.
        - it actually makes the train slower too :)
        - but it makes sampling faster
- now, we can make it faster!!!
    - in fact, if we can make sampling faster we might be able to increase n_steps and bacth size too 